# 04 — Trader insights (Genie)

Creates **Unity Catalog metric views** and provisions the Genie Space **Energy Trading Genie - Short Term**
over near-delivery, control-tower, and DSR gold tables (notebooks 01 → 03 → 02). Sample questions live in the Space config only.

Spec: [`../specifications/04-trader-insights-genie.md`](../specifications/04-trader-insights-genie.md)

**Depends on:** notebooks `01` → `03` → `02` (all `short_term_*` gold tables materialised).

**Produces**
- UC metric views `mv_st_*` on key gold tables
- Genie Space **Energy Trading Genie - Short Term** (REST / SDK; `space_id` printed in notebook output)


In [ ]:
# Databricks notebook source
import os
from pathlib import Path

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))
dbutils.widgets.text("warehouse_id", "")

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
WAREHOUSE_WIDGET = dbutils.widgets.get("warehouse_id").strip()

spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}` COMMENT 'Energy trading demo data'")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")

def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

print(f"Target: {CATALOG}.{SCHEMA}")


In [ ]:
REQUIRED = [
    "short_term_gold_squaring_actions",
    "short_term_gold_portfolio_balance",
    "short_term_gold_control_tower_summary",
    "short_term_gold_dsr_summary",
    "short_term_gold_dsr_dispatch",
]
missing = [t for t in REQUIRED if not spark.catalog.tableExists(fq(t))]
assert not missing, f"Run notebooks 01 → 03 → 02 first; missing: {missing}"


In [ ]:
_cfg_paths = [Path.cwd() / "genie_space_config.py"]
try:
    _nb = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    )
    _cfg_paths.insert(0, Path(_nb).parent / "genie_space_config.py")
except Exception:
    pass
_cfg_py = next((p for p in _cfg_paths if p.is_file()), None)
if _cfg_py is None:
    raise FileNotFoundError("genie_space_config.py not found next to this notebook")
_fq_table = fq  # notebook helper; genie_space_config must not overwrite fq()
exec(_cfg_py.read_text(), globals())
fq = _fq_table
print("Loaded", _cfg_py.name)


## 1. Unity Catalog metric views (optional)

Governed KPIs for Genie — one metric view per primary gold table. Works on **serverless** when
metric views are enabled (Runtime 17.2+ YAML). Failures are non-fatal; Genie uses base tables.

In [ ]:
created_mvs = []
mv_errors = []
for spec in METRIC_VIEW_SPECS:
    sql = build_metric_view_sql(CATALOG, SCHEMA, spec)
    try:
        spark.sql(sql)
        created_mvs.append(spec[0])
    except Exception as exc:
        mv_errors.append((spec[0], str(exc)))

print("Metric views created:", created_mvs)
if mv_errors:
    print("Metric view errors (see message — Genie still uses base tables):")
    for name, err in mv_errors:
        print(f"  {name}: {err[:200]}")


## 2. Create or update Genie Space

Uses `provision_genie_space()` (REST or SDK). Set widget `warehouse_id` to a SQL warehouse id, or leave blank to auto-pick a running warehouse.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import State

w = WorkspaceClient()


def _pick_warehouse() -> str:
    if WAREHOUSE_WIDGET:
        return WAREHOUSE_WIDGET
    running = []
    other = []
    for wh in w.warehouses.list():
        wid = (wh.id or "").strip()
        if not wid:
            continue
        if wh.state == State.RUNNING:
            running.append(wid)
        else:
            other.append(wid)
    if running:
        return running[0]
    if other:
        return other[0]
    raise RuntimeError("No SQL warehouse found in workspace — create one and set warehouse_id widget")


warehouse_id = _pick_warehouse()
include_mvs = len(created_mvs) == len(METRIC_VIEW_SPECS)
serialized = build_serialized_space(CATALOG, SCHEMA, include_metric_views=include_mvs)

space, operation = provision_genie_space(
    w,
    title=GENIE_SPACE_TITLE,
    description=GENIE_SPACE_DESCRIPTION,
    warehouse_id=warehouse_id,
    serialized_space=serialized,
)

space_id = space.space_id
print(f"Genie Space {operation}: {GENIE_SPACE_TITLE}")
print(f"  space_id:     {space_id}")
print(f"  warehouse_id: {warehouse_id}")
print(f"  tables:       {len(GENIE_TABLES)}")
print(f"  metric_views: {len(created_mvs) if include_mvs else 0}")


In [ ]:
for t in list(created_mvs):
    name = t if t.startswith("short_term") or t.startswith("mv_st_") else t
    if spark.catalog.tableExists(fq(name)):
        n = spark.table(fq(name)).count()
        print(f"  {name:42s}  {n:>10,} rows")


## Unity Catalog comments

In [ ]:
from pathlib import Path

_uc_paths = [Path.cwd() / "uc_table_comments.py"]
try:
    _nb = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    )
    _uc_paths.insert(0, Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found")
exec(_uc_py.read_text(), globals())
apply_short_term_notebook_04_comments(spark, CATALOG, SCHEMA)
skipped = [n for n in _NOTEBOOK_04 if not spark.catalog.tableExists(fq(n))]
if skipped:
    print("UC comments skipped (object not created):", skipped)
print("UC comments applied.")
